# 07 - Final-report extensions: separated training / test evaluation, tuning, error analysis

This notebook adds the evidence the final report needs on top of notebooks 03 to 06:

1. **RQ1** - the same four models as notebook 03 are refitted with the same seed and split, so the test results must
   reproduce notebook 03; training results are added next to them (over-fitting check), a tuned version of the two
   machine-learning models is evaluated, and the best model is examined with confusion matrices and a risk-decile (lift) table.
2. **RQ2** - the regression is fitted on 80% of the routes and scored on the remaining 20% (a separated evaluation for an inferential model).
3. **RQ3** - cluster stability under bootstrap resampling (adjusted Rand index), because clustering has no train/test accuracy.

The tuned model and the feature specification are saved to `app/model/` for the deployment prototype.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
from labels import MONTH_NAMES, DOW_NAMES, carrier_name, airport_name
import json, time
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import xgboost as xgb
from metrics import classification_report_dict, confusion_counts
from plot_style import set_style, save_fig
from rq1_features import fit_design
set_style()
T0 = time.time()

## 1. RQ1 - rebuild the data, split and models exactly as in notebook 03

In [2]:
raw = pd.read_csv(PROCESSED / "rq1_sample.csv.gz")
df, X, y, spec = fit_design(raw, TOP_AIRPORTS_RQ1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
prev = float(y_tr.mean())
print(f"{len(df):,} flights | train {len(X_tr):,} | test {len(X_te):,} | threshold (training prevalence) {prev:.4f} | design columns {X.shape[1]}")

Xc_tr = sm.add_constant(X_tr.astype("float64")); Xc_te = sm.add_constant(X_te.astype("float64"), has_constant="add")
logit = sm.Logit(y_tr, Xc_tr).fit(method="newton", maxiter=100, disp=0)
rf = RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=100, n_jobs=-1, random_state=RANDOM_STATE).fit(X_tr, y_tr)
xg = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                       tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, eval_metric="auc").fit(X_tr, y_tr)
P = {   # name -> (train probabilities, test probabilities)
    "Logistic regression": (np.asarray(logit.predict(Xc_tr)), np.asarray(logit.predict(Xc_te))),
    "Random forest": (rf.predict_proba(X_tr)[:, 1], rf.predict_proba(X_te)[:, 1]),
    "XGBoost": (xg.predict_proba(X_tr)[:, 1], xg.predict_proba(X_te)[:, 1]),
}
stored = {r["Model"]: r for r in load_results()["rq1"]["model_comparison"]}
for k, (ptr, pte) in P.items():
    print(f"{k:20s} test AUC now {roc_auc_score(y_te, pte):.4f} | stored in notebook 03: {stored[k]['auc']:.4f}")

400,000 flights | train 320,000 | test 80,000 | threshold (training prevalence) 0.2158 | design columns 102


Logistic regression  test AUC now 0.6647 | stored in notebook 03: 0.6647
Random forest        test AUC now 0.6515 | stored in notebook 03: 0.6515
XGBoost              test AUC now 0.6775 | stored in notebook 03: 0.6775


## 2. Hyper-parameter tuning (random search, 3-fold cross-validation on a 90,000-flight subsample of the training set)
Tuning uses training data only; the test set is touched once, for the final score.

In [3]:
n_sub = 90_000
idx = np.random.RandomState(RANDOM_STATE).choice(len(X_tr), n_sub, replace=False)
Xs, ys = X_tr.iloc[idx], y_tr[idx]
skf = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)

xgb_space = {"max_depth": [3, 4, 5, 6, 8], "learning_rate": [0.03, 0.05, 0.1], "n_estimators": [200, 300, 500],
             "min_child_weight": [1, 5, 20, 50], "subsample": [0.7, 0.8, 1.0], "colsample_bytree": [0.5, 0.8, 1.0], "reg_lambda": [1, 5, 20]}
s1 = RandomizedSearchCV(xgb.XGBClassifier(tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE), xgb_space, n_iter=14,
                        scoring="roc_auc", cv=skf, random_state=RANDOM_STATE, n_jobs=1).fit(Xs, ys)
print("XGBoost best", s1.best_params_, round(s1.best_score_, 4))

rf_space = {"max_depth": [8, 12, 16, 20], "min_samples_leaf": [20, 50, 100, 200], "max_features": ["sqrt", 0.3, 0.5], "n_estimators": [100, 150]}
s2 = RandomizedSearchCV(RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE), rf_space, n_iter=8,
                        scoring="roc_auc", cv=skf, random_state=RANDOM_STATE, n_jobs=1).fit(Xs, ys)
print("Random forest best", s2.best_params_, round(s2.best_score_, 4))
print(f"elapsed {time.time()-T0:.0f} s")

XGBoost best {'subsample': 0.7, 'reg_lambda': 5, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8} 0.6635


Random forest best {'n_estimators': 150, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': 20} 0.6546
elapsed 103 s


In [4]:
xg_t = xgb.XGBClassifier(tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, **s1.best_params_).fit(X_tr, y_tr)
rf_t = RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE, **s2.best_params_).fit(X_tr, y_tr)
P["XGBoost (tuned)"] = (xg_t.predict_proba(X_tr)[:, 1], xg_t.predict_proba(X_te)[:, 1])
P["Random forest (tuned)"] = (rf_t.predict_proba(X_tr)[:, 1], rf_t.predict_proba(X_te)[:, 1])

rows = []
base_te = float(1 - y_te.mean()); base_tr = float(1 - y_tr.mean())
rows.append({"Model": "Majority-class baseline", "train_accuracy": base_tr, "test_accuracy": base_te, "train_auc": 0.5, "test_auc": 0.5,
             "train_f1": 0.0, "test_f1": 0.0, "test_precision": 0.0, "test_recall": 0.0})
for k, (ptr, pte) in P.items():
    a, b = classification_report_dict(y_tr, ptr, prev), classification_report_dict(y_te, pte, prev)
    rows.append({"Model": k, "train_accuracy": a["accuracy"], "test_accuracy": b["accuracy"], "train_auc": a["auc"], "test_auc": b["auc"],
                 "train_f1": a["f1"], "test_f1": b["f1"], "test_precision": b["precision"], "test_recall": b["recall"]})
tt = pd.DataFrame(rows); tt["auc_gap"] = tt["train_auc"] - tt["test_auc"]
tt.to_csv(TAB / "rq1_train_test.csv", index=False)
tt.round(4)

,Model,train_accuracy,test_accuracy,train_auc,test_auc,train_f1,test_f1,test_precision,test_recall,auc_gap
0,Majority-class baseline,0.7842,0.7842,0.5000,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000
1,Logistic regression,0.6015,0.6032,0.6622,0.6647,0.4114,0.4129,0.3033,0.6466,-0.0026
2,Random forest,0.5909,0.5892,0.6554,0.6515,0.4056,0.4033,0.2937,0.6433,0.0038
3,XGBoost,0.6441,0.6237,0.7199,0.6775,0.4543,0.4229,0.3161,0.6388,0.0424
4,XGBoost (tuned),0.6222,0.6145,0.6867,0.6724,0.4287,0.4168,0.3094,0.6382,0.0143
5,Random forest (tuned),0.6102,0.5995,0.6845,0.6609,0.4241,0.4089,0.3000,0.6419,0.0237


## 3. Best model: confusion matrices, risk deciles, and the tuned parameters

In [5]:
best = tt[tt["Model"] != "Majority-class baseline"].sort_values("test_auc", ascending=False).iloc[0]["Model"]
print("best model on the test set:", best)
cm = {k: confusion_counts(y_te, P[k][1], prev) for k in ("Logistic regression", best)}

fig, axs = plt.subplots(1, 2, figsize=(6.5, 2.9))
for ax, (k, c) in zip(axs, cm.items()):
    M = np.array([[c["tn"], c["fp"]], [c["fn"], c["tp"]]])
    share = M / M.sum(axis=1, keepdims=True)
    ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{M[i, j]:,}\n({share[i, j]*100:.1f}% of row)", ha="center", va="center", fontsize=8,
                    color="white" if share[i, j] > 0.55 else INK)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted\non time", "Predicted\nlate"], fontsize=8)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Actually\non time", "Actually\nlate"], fontsize=8)
    ax.set_title(k, fontsize=9, loc="left"); ax.grid(False)
fig.tight_layout(); save_fig(fig, "fig21_rq1_confusion")

# risk deciles: observed late rate in each tenth of test flights ranked by predicted probability
pb = P[best][1]
dec = pd.DataFrame({"p": pb, "y": y_te}); dec["decile"] = pd.qcut(dec["p"].rank(method="first"), 10, labels=False) + 1
lift = dec.groupby("decile").agg(flights=("y", "size"), predicted=("p", "mean"), observed=("y", "mean"))
lift["observed"] *= 100; lift["predicted"] *= 100; lift["lift"] = lift["observed"] / (100 * y_te.mean())
lift.to_csv(TAB / "rq1_risk_deciles.csv")
fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.bar(lift.index, lift["observed"], color=PALETTE[0], width=0.7)
ax.axhline(100 * y_te.mean(), color=INK_2, ls="--", lw=1); ax.text(0.6, 100 * y_te.mean() + 0.6, f"All flights: {100*y_te.mean():.1f}%", fontsize=8, color=INK)
ax.set_xticks(range(1, 11)); ax.set_xlabel("Predicted-risk decile (1 = lowest risk, 10 = highest risk)"); ax.set_ylabel("Flights that arrived late (%)")
fig.tight_layout(); save_fig(fig, "fig22_rq1_deciles")
lift.round(2)

best model on the test set: XGBoost


,flights,predicted,observed,lift
decile,,,,
1,8000,7.510000,7.38,0.34
2,8000,11.120000,10.76,0.50
3,8000,13.780000,13.26,0.61
4,8000,16.250000,15.62,0.72
5,8000,18.680000,17.76,0.82
6,8000,21.209999,21.20,0.98
7,8000,24.170000,24.69,1.14
8,8000,27.820000,27.38,1.27
9,8000,32.709999,32.96,1.53


In [6]:
pr = {"xgb": s1.best_params_, "xgb_cv_auc": float(s1.best_score_), "rf": s2.best_params_, "rf_cv_auc": float(s2.best_score_)}
pr = json.loads(json.dumps(pr, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
imp = pd.Series(xg_t.feature_importances_, index=X.columns).groupby(pd.Series(spec["col_to_factor"])).sum().sort_values(ascending=False)
save_result("rq1_extension", {
    "best_model": best, "tuning": pr, "threshold": prev,
    "table": tt.round(5).to_dict("records"), "confusion": cm,
    "deciles": lift.round(4).reset_index().to_dict("records"),
    "top_decile_late_pct": float(lift.loc[10, "observed"]), "bottom_decile_late_pct": float(lift.loc[1, "observed"]),
    "top_decile_lift": float(lift.loc[10, "lift"]),
    "n_features": int(X.shape[1]), "gain_importance_by_factor": imp.round(4).to_dict()})
# save the tuned model + feature specification for the deployment prototype
APP = ROOT / "app" / "model"; APP.mkdir(parents=True, exist_ok=True)
(xg_t if best == "XGBoost (tuned)" else xg).save_model(str(APP / "late_arrival_xgboost.json"))
json.dump({**spec, "threshold": prev, "model": best, "test_auc": float(roc_auc_score(y_te, P[best][1])),
           "test_prevalence": float(y_te.mean())}, open(APP / "feature_spec.json", "w"), indent=1)
print("saved model for deployment:", best, f"| elapsed {time.time()-T0:.0f} s")

saved model for deployment: XGBoost | elapsed 117 s


## 3b. Comparison with current practice
Travellers and airline schedulers today mostly rely on *historical on-time percentages* by airline and route, which is what the
public Bureau of Transportation Statistics tables and travel websites report. Two such benchmarks are built from the training flights only
(smoothed towards the parent group so rare routes do not get extreme rates) and scored on the same test flights.

In [7]:
M_SMOOTH = 20
trn_df, tst_df = df.loc[X_tr.index], df.loc[X_te.index]
g0 = float(trn_df["is_late"].mean())
car = trn_df.groupby("carrier")["is_late"].agg(["sum", "count"])
p_car = (car["sum"] + M_SMOOTH * g0) / (car["count"] + M_SMOOTH)
bench_car = tst_df["carrier"].map(p_car).fillna(g0).values
rt = trn_df.groupby(["carrier", "origin", "dest"])["is_late"].agg(["sum", "count"]).reset_index()
rt["parent"] = rt["carrier"].map(p_car).fillna(g0)
rt["p"] = (rt["sum"] + M_SMOOTH * rt["parent"]) / (rt["count"] + M_SMOOTH)
key = tst_df[["carrier", "origin", "dest"]].merge(rt[["carrier", "origin", "dest", "p"]], how="left", on=["carrier", "origin", "dest"])
bench_rt = key["p"].fillna(tst_df["carrier"].map(p_car).fillna(g0).reset_index(drop=True)).values
cur = {"Historical late rate by airline": float(roc_auc_score(y_te, bench_car)),
       "Historical late rate by airline and route": float(roc_auc_score(y_te, bench_rt))}
print(cur)
print("share of test flights whose airline-route pair appears in the training flights:", round(float(key["p"].notna().mean()) * 100, 1), "%")
save_result("rq1_current_practice", {"auc": cur, "smoothing": M_SMOOTH, "route_coverage_pct": float(key["p"].notna().mean() * 100)})

{'Historical late rate by airline': 0.5476486194705474, 'Historical late rate by airline and route': 0.5726487566899052}
share of test flights whose airline-route pair appears in the training flights: 99.7 %


{'auc': {'Historical late rate by airline': 0.5476486194705474,
  'Historical late rate by airline and route': 0.5726487566899052},
 'smoothing': 20,
 'route_coverage_pct': 99.67375000000001}

## 4. RQ2 - separated evaluation: fit on 80% of the carrier-routes, score on the other 20%
Splitting by route (not by row) keeps all months of one route on the same side, so the test routes are genuinely unseen.

In [8]:
rma = pd.read_csv(PROCESSED / "route_month_analysis.csv")
rma["route"] = rma["carrier_id"].astype(str) + "_" + rma["origin_id"].astype(str) + "_" + rma["dest_id"].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
itr, ite = next(gss.split(rma, groups=rma["route"]))
a, b = rma.iloc[itr], rma.iloc[ite]
m = sm.OLS(a["late_rate"], sm.add_constant(a["load_factor"])).fit()
def r2(d):
    pred = m.params["const"] + m.params["load_factor"] * d["load_factor"]
    return float(1 - ((d["late_rate"] - pred) ** 2).sum() / ((d["late_rate"] - d["late_rate"].mean()) ** 2).sum())
def rmse_(d, base=False):
    pred = (a["late_rate"].mean() if base else m.params["const"] + m.params["load_factor"] * d["load_factor"])
    return float(np.sqrt(((d["late_rate"] - pred) ** 2).mean()))
rq2x = {"n_train": int(len(a)), "n_test": int(len(b)), "routes_train": int(a["route"].nunique()), "routes_test": int(b["route"].nunique()),
        "slope_train": float(m.params["load_factor"]), "r2_train": r2(a), "r2_test": r2(b),
        "rmse_train": rmse_(a), "rmse_test": rmse_(b), "rmse_test_mean_only": rmse_(b, base=True),
        "pearson_test": float(stats.pearsonr(b["load_factor"], b["late_rate"])[0])}
save_result("rq2_extension", rq2x); rq2x

{'n_train': 118488,
 'n_test': 29753,
 'routes_train': 7998,
 'routes_test': 2000,
 'slope_train': 0.11700632975014738,
 'r2_train': 0.011451385909705514,
 'r2_test': 0.012332329308301082,
 'rmse_train': 11.270269550813156,
 'rmse_test': 11.16722788506904,
 'rmse_test_mean_only': 11.236733626858985,
 'pearson_test': 0.11116817327859194}

## 5. RQ3 - cluster stability
Clusters are refitted on 200 bootstrap resamples of the carrier-airport units; each refit is compared with the original clustering on all units with the adjusted Rand index (1 = identical grouping, 0 = chance).

In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
ca = pd.read_csv(PROCESSED / "carrier_airport.csv")
feats = ["avg_load_factor", "mean_monthly_departures", "avg_distance", "seasonal_cv"]
Z = ca[feats].copy(); Z["mean_monthly_departures"] = np.log1p(Z["mean_monthly_departures"])
Zs = StandardScaler().fit_transform(Z)
k = load_results()["rq3"]["k"]
ref = KMeans(k, n_init=50, random_state=RANDOM_STATE).fit(Zs)
rng = np.random.RandomState(RANDOM_STATE); aris = []
for _ in range(200):
    ix = rng.choice(len(Zs), len(Zs), replace=True)
    km = KMeans(k, n_init=10, random_state=int(rng.randint(1e6))).fit(Zs[ix])
    aris.append(adjusted_rand_score(ref.labels_, km.predict(Zs)))
rq3x = {"k": int(k), "bootstrap_runs": 200, "ari_mean": float(np.mean(aris)), "ari_sd": float(np.std(aris)),
        "ari_p5": float(np.percentile(aris, 5)), "ari_min": float(np.min(aris))}
save_result("rq3_extension", rq3x); rq3x

{'k': 4,
 'bootstrap_runs': 200,
 'ari_mean': 0.9079838820662662,
 'ari_sd': 0.06931277146561189,
 'ari_p5': 0.7674308328691711,
 'ari_min': 0.5885856991811712}